In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from collections import Counter
from functools import reduce
from datetime import datetime

SILVER_CATALOG = "cms_beneficiary"
SILVER_SCHEMA  = "silver"
SILVER_FQ      = f"{SILVER_CATALOG}.{SILVER_SCHEMA}"
BRONZE_VIEW    = "cms_beneficiary.bronze.cms_beneficiary_all"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {SILVER_CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {SILVER_FQ}")
print("Setup complete. Silver target:", SILVER_FQ)

In [0]:
def assert_unique_columns(df, name="df"):
    """Fail loudly if a DataFrame has duplicate column names."""
    dupes = [c for c, n in Counter(df.columns).items() if n > 1]
    if dupes:
        raise ValueError(f"{name} has duplicate columns: {dupes}")
    return df


MONTHLY_PREFIXES = [
    "MDCR_STATUS_CODE_",
    "MDCR_ENTLMT_BUYIN_IND_",
    "HMO_IND_",
    "PTC_CNTRCT_ID_",
    "PTC_PBP_ID_",
    "PTC_PLAN_TYPE_CD_",
    "PTD_CNTRCT_ID_",
    "PTD_PBP_ID_",
    "PTD_SGMT_ID_",
    "RDS_IND_",
    "DUAL_STUS_CD_",
    "CST_SHR_GRP_CD_",
    "STATE_CNTY_FIPS_CD_",
]

def is_monthly(col: str) -> bool:
    return any(col.startswith(p) and col[-2:].isdigit() for p in MONTHLY_PREFIXES)


INT_COLS = {
    "SEX_IDENT_CD", "BENE_RACE_CD",
    "ENTLMT_RSN_ORIG", "ENTLMT_RSN_CURR",
    "BENE_ENROLLMT_REF_YR",
    "BENE_HI_CVRAGE_TOT_MONS", "BENE_SMI_CVRAGE_TOT_MONS",
    "BENE_STATE_BUYIN_TOT_MONS", "BENE_HMO_CVRAGE_TOT_MONS",
    "RDS_CVRG_MONS", "DUAL_ELGBL_MONS", "PTD_PLAN_CVRG_MONS",
    "AGE_AT_END_REF_YR",
}

DATE_COLS = {"BENE_BIRTH_DT", "BENE_DEATH_DT", "COVSTART"}

META_COLS = {"_ingested_at", "_source_file", "_ref_year", "_corrupt_record"}

print("Helpers loaded.")

In [0]:
bronze = spark.table(BRONZE_VIEW)

all_cols     = bronze.columns
monthly_cols = [c for c in all_cols if is_monthly(c)]
scalar_cols  = [c for c in all_cols
                if c not in monthly_cols and c not in META_COLS]

print(f"Bronze rows    : {bronze.count():,}")
print(f"Bronze columns : {len(all_cols):,}")
print(f"Monthly columns: {len(monthly_cols):,}")
print(f"Scalar columns : {len(scalar_cols):,}")

assert len(set(scalar_cols)) == len(scalar_cols), "scalar_cols has duplicates"
assert len(set(monthly_cols)) == len(monthly_cols), "monthly_cols has duplicates"
assert len(set(scalar_cols) & set(monthly_cols)) == 0, "scalar/monthly overlap"

In [0]:
core = bronze.select(*scalar_cols, "_ref_year", "_ingested_at", "_source_file")
core = assert_unique_columns(core, "core_raw")

string_cols = [c for c, t in core.dtypes if t == "string"]
core = core.select(*[
    F.trim(F.col(c)).alias(c) if c in string_cols else F.col(c)
    for c in core.columns
])

for c in INT_COLS:
    if c in core.columns:
        core = core.withColumn(c, F.col(c).cast(IntegerType()))

for c in DATE_COLS:
    if c in core.columns:
        core = core.withColumn(c, F.to_date(F.col(c), "dd-MMM-yyyy"))

core = (core
    .withColumn("age_band",
        F.when(F.col("AGE_AT_END_REF_YR") < 65, "under_65")
         .when(F.col("AGE_AT_END_REF_YR") < 75, "65_74")
         .when(F.col("AGE_AT_END_REF_YR") < 85, "75_84")
         .otherwise("85_plus"))
    .withColumn("is_deceased",
        F.when(F.col("BENE_DEATH_DT").isNotNull(), True).otherwise(False)))

core = core.dropDuplicates(["BENE_ID", "_ref_year"])

core = core.withColumn("_silver_processed_at", F.current_timestamp())

core = assert_unique_columns(core, "core")
print(f"core rows    : {core.count():,}")
print(f"core columns : {len(core.columns):,}")

In [0]:
def melt_group(bronze_df, prefix: str, metric_name: str):
    """Melt a 12-month group of columns into long format."""
    cols = [f"{prefix}{i:02d}" for i in range(1, 13)]
    present = [c for c in cols if c in bronze_df.columns]
    if not present:
        return None

    df = (bronze_df
          .select("BENE_ID", "_ref_year", *present)
          .melt(ids=["BENE_ID", "_ref_year"],
                values=present,
                variableColumnName="month_col",
                valueColumnName=metric_name)
          .withColumn("month_num",
                      F.regexp_extract("month_col", r"(\d+)$", 1).cast("int"))
          .drop("month_col")
          .filter(F.col(metric_name).isNotNull()))
    return df


METRIC_GROUPS = [
    ("HMO_IND_",               "hmo_ind"),
    ("MDCR_STATUS_CODE_",      "mdcr_status_code"),
    ("MDCR_ENTLMT_BUYIN_IND_", "mdcr_buyin_ind"),
    ("DUAL_STUS_CD_",          "dual_status_code"),
    ("RDS_IND_",               "rds_ind"),
    ("CST_SHR_GRP_CD_",        "cost_share_group"),
    ("PTC_CNTRCT_ID_",         "ptc_contract_id"),
    ("PTC_PBP_ID_",            "ptc_pbp_id"),
    ("PTC_PLAN_TYPE_CD_",      "ptc_plan_type"),
    ("PTD_CNTRCT_ID_",         "ptd_contract_id"),
    ("PTD_PBP_ID_",            "ptd_pbp_id"),
    ("PTD_SGMT_ID_",           "ptd_segment_id"),
    ("STATE_CNTY_FIPS_CD_",    "state_county_fips"),
]

monthly_frames = []
for prefix, name in METRIC_GROUPS:
    f = melt_group(bronze, prefix, name)
    if f is not None:
        monthly_frames.append(f)
        print(f"  ✓ melted {name:20s} ({f.count():,} rows)")
    else:
        print(f"  – skipped {name} (no columns)")

monthly = reduce(
    lambda a, b: a.join(b, on=["BENE_ID", "_ref_year", "month_num"], how="outer"),
    monthly_frames,
)

monthly = monthly.withColumn("_silver_processed_at", F.current_timestamp())
monthly = assert_unique_columns(monthly, "monthly")

print(f"\nmonthly rows    : {monthly.count():,}")
print(f"monthly columns : {len(monthly.columns):,}")

In [0]:
def build_dq_row(df, table_name: str):
    total         = df.count()
    null_id       = df.filter(F.col("BENE_ID").isNull()).count()
    null_yr       = df.filter(F.col("_ref_year").isNull()).count()
    distinct_key  = df.select("BENE_ID", "_ref_year").distinct().count()
    duplicate_key = total - distinct_key
    return (table_name, datetime.utcnow(), int(total),
            int(null_id), int(null_yr), int(duplicate_key))


dq_schema = """
    table_name        string,
    run_ts            timestamp,
    row_count         long,
    null_bene_id      long,
    null_ref_year     long,
    duplicate_key_cnt long
"""

dq_rows = [
    build_dq_row(core,    "cms_beneficiary_core"),
    build_dq_row(monthly, "cms_beneficiary_monthly"),
]

dq_df = spark.createDataFrame(dq_rows, schema=dq_schema)
display(dq_df)

In [0]:
def write_silver(df, name: str, partition_col: str = "_ref_year"):
    tgt = f"{SILVER_FQ}.{name}"
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option("mergeSchema", "true")
        .partitionBy(partition_col)
        .saveAsTable(tgt))
    print(f"✓ wrote {tgt}  ({spark.table(tgt).count():,} rows)")


write_silver(core,    "cms_beneficiary_core")
write_silver(monthly, "cms_beneficiary_monthly")

(dq_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{SILVER_FQ}.cms_beneficiary_dq"))
print(f"✓ appended to {SILVER_FQ}.cms_beneficiary_dq")

In [0]:
%sql
SELECT  _ref_year,
        COUNT(*)                                     AS core_rows,
        COUNT(DISTINCT BENE_ID)                      AS unique_benes,
        ROUND(AVG(AGE_AT_END_REF_YR), 1)             AS avg_age,
        SUM(CASE WHEN is_deceased THEN 1 ELSE 0 END) AS deceased
FROM    cms_beneficiary.silver.cms_beneficiary_core
GROUP BY _ref_year
ORDER BY _ref_year;

In [0]:
%sql
SELECT  _ref_year,
        COUNT(*)                    AS fact_rows,
        COUNT(DISTINCT BENE_ID)     AS benes,
        COUNT(DISTINCT month_num)   AS months_covered
FROM    cms_beneficiary.silver.cms_beneficiary_monthly
GROUP BY _ref_year
ORDER BY _ref_year;

In [0]:
%sql
SELECT  table_name,
        run_ts,
        row_count,
        null_bene_id,
        null_ref_year,
        duplicate_key_cnt,
        ROUND(100.0 * duplicate_key_cnt / NULLIF(row_count,0), 2) AS dup_pct
FROM    cms_beneficiary.silver.cms_beneficiary_dq
ORDER BY run_ts DESC;

In [0]:
%sql
SELECT  _ref_year,
        STATE_CODE,
        age_band,
        COUNT(*) AS benes
FROM    cms_beneficiary.silver.cms_beneficiary_core
WHERE   _ref_year IN (2015, 2025)
GROUP BY _ref_year, STATE_CODE, age_band
ORDER BY _ref_year, benes DESC
LIMIT 40;

In [0]:
%sql
SELECT  _ref_year,
        STATE_CODE,
        age_band,
        COUNT(*) AS benes
FROM    cms_beneficiary.silver.cms_beneficiary_core
WHERE   _ref_year IN (2015, 2025)
GROUP BY _ref_year, STATE_CODE, age_band
ORDER BY _ref_year, benes DESC
LIMIT 40;

In [0]:
%sql
SELECT  _ref_year,
        COUNT(DISTINCT BENE_ID)    AS total_benes,
        COUNT(DISTINCT CASE WHEN dual_status_code <> 'NA'
                             AND dual_status_code IS NOT NULL
                            THEN BENE_ID END) AS dual_benes,
        ROUND(100.0 *
              COUNT(DISTINCT CASE WHEN dual_status_code <> 'NA'
                                   AND dual_status_code IS NOT NULL
                                  THEN BENE_ID END)
              / COUNT(DISTINCT BENE_ID), 2)   AS dual_share_pct
FROM    cms_beneficiary.silver.cms_beneficiary_monthly
GROUP BY _ref_year
ORDER BY _ref_year;